# Classification

# Data Loading & Preprocessing

Import required libraries

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (
    accuracy_score,
    confusion_matrix,
    classification_report
)


Load dataset

In [15]:
df = pd.read_csv("breast_cancer.csv")

Display first rows

In [16]:
df.head()

,id,diagnosis,diagnosis_Num,radius_mean,texture_mean,perimeter_mean,area_mean,smoothness_mean,compactness_mean,concavity_mean,...,radius_worst,texture_worst,perimeter_worst,area_worst,smoothness_worst,compactness_worst,concavity_worst,concave points_worst,symmetry_worst,fractal_dimension_worst
0,842302,M,1,17.99,10.38,122.80,1001.0,0.11840,0.27760,0.3001,...,25.38,17.33,184.60,2019.0,0.1622,0.6656,0.7119,0.2654,0.4601,0.11890
1,842517,M,1,20.57,17.77,132.90,1326.0,0.08474,0.07864,0.0869,...,24.99,23.41,158.80,1956.0,0.1238,0.1866,0.2416,0.1860,0.2750,0.08902
2,84300903,M,1,19.69,21.25,130.00,1203.0,0.10960,0.15990,0.1974,...,23.57,25.53,152.50,1709.0,0.1444,0.4245,0.4504,0.2430,0.3613,0.08758
3,84348301,M,1,11.42,20.38,77.58,386.1,0.14250,0.28390,0.2414,...,14.91,26.50,98.87,567.7,0.2098,0.8663,0.6869,0.2575,0.6638,0.17300
4,84358402,M,1,20.29,14.34,135.10,1297.0,0.10030,0.13280,0.1980,...,22.54,16.67,152.20,1575.0,0.1374,0.2050,0.4000,0.1625,0.2364,0.07678


# Data Preprocessing

Drop Irrelevant Columns

In [17]:
df = df.drop(['id', 'diagnosis'], axis=1)


Separate features and target

In [18]:
X = df.drop('diagnosis_Num', axis=1)
y = df['diagnosis_Num']


Train-test split

In [19]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)


Feature Scaling

In [20]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)


Model 1 – Logistic Regression

Baseline Model

In [21]:
log_reg = LogisticRegression(max_iter=500)
log_reg.fit(X_train_scaled, y_train)

y_pred_lr = log_reg.predict(X_test_scaled)
accuracy_lr = accuracy_score(y_test, y_pred_lr)

accuracy_lr


0.9649122807017544

Hyperparameter Tuning (C)

In [22]:
param_grid_lr = {
    'C': [0.01, 0.1, 1, 10]
}

grid_lr = GridSearchCV(
    LogisticRegression(max_iter=500),
    param_grid_lr,
    cv=5,
    scoring='accuracy'
)

grid_lr.fit(X_train_scaled, y_train)


GridSearchCV(cv=5, estimator=LogisticRegression(max_iter=500),
             param_grid={'C': [0.01, 0.1, 1, 10]}, scoring='accuracy')

In [23]:
best_lr = grid_lr.best_estimator_

y_pred_lr_tuned = best_lr.predict(X_test_scaled)
accuracy_lr_tuned = accuracy_score(y_test, y_pred_lr_tuned)

accuracy_lr_tuned


0.9649122807017544

Model 2 – Support Vector Machine (RBF Kernel)

Baseline Model

In [24]:
svm = SVC(kernel='rbf')
svm.fit(X_train_scaled, y_train)

y_pred_svm = svm.predict(X_test_scaled)
accuracy_svm = accuracy_score(y_test, y_pred_svm)

accuracy_svm


0.9736842105263158

Hyperparameter Tuning (C, Gamma)

In [25]:
param_grid_svm = {
    'C': [0.1, 1, 10],
    'gamma': ['scale', 0.01, 0.001]
}

grid_svm = GridSearchCV(
    SVC(kernel='rbf'),
    param_grid_svm,
    cv=5,
    scoring='accuracy',
    n_jobs=-1
)

grid_svm.fit(X_train_scaled, y_train)


GridSearchCV(cv=5, estimator=SVC(), n_jobs=-1,
             param_grid={'C': [0.1, 1, 10], 'gamma': ['scale', 0.01, 0.001]},
             scoring='accuracy')

In [26]:
best_svm = grid_svm.best_estimator_

y_pred_svm_tuned = best_svm.predict(X_test_scaled)
accuracy_svm_tuned = accuracy_score(y_test, y_pred_svm_tuned)

accuracy_svm_tuned


0.9736842105263158

Model 3 – Neural Network Classifier

In [27]:
mlp = MLPClassifier(
    hidden_layer_sizes=(64, 32),
    activation='relu',
    solver='adam',
    max_iter=500,
    random_state=42
)

mlp.fit(X_train_scaled, y_train)


MLPClassifier(hidden_layer_sizes=(64, 32), max_iter=500, random_state=42)

In [28]:
y_pred_mlp = mlp.predict(X_test_scaled)
accuracy_mlp = accuracy_score(y_test, y_pred_mlp)

accuracy_mlp


0.9649122807017544

# Model Evaluation

Confusion Matrix & Classification Report

In [29]:
models = {
    "Logistic Regression (Tuned)": y_pred_lr_tuned,
    "SVM (Tuned)": y_pred_svm_tuned,
    "Neural Network": y_pred_mlp
}

for model_name, predictions in models.items():
    print(f"\n{model_name}")
    print(confusion_matrix(y_test, predictions))
    print(classification_report(y_test, predictions))



Logistic Regression (Tuned)
[[71  1]
 [ 3 39]]
              precision    recall  f1-score   support

           0       0.96      0.99      0.97        72
           1       0.97      0.93      0.95        42

    accuracy                           0.96       114
   macro avg       0.97      0.96      0.96       114
weighted avg       0.97      0.96      0.96       114


SVM (Tuned)
[[72  0]
 [ 3 39]]
              precision    recall  f1-score   support

           0       0.96      1.00      0.98        72
           1       1.00      0.93      0.96        42

    accuracy                           0.97       114
   macro avg       0.98      0.96      0.97       114
weighted avg       0.97      0.97      0.97       114


Neural Network
[[72  0]
 [ 4 38]]
              precision    recall  f1-score   support

           0       0.95      1.00      0.97        72
           1       1.00      0.90      0.95        42

    accuracy                           0.96       114
   macro avg 

Model Performance Comparison

In [30]:
results = pd.DataFrame({
    "Model": [
        "Logistic Regression (Tuned)",
        "SVM (Tuned)",
        "Neural Network"
    ],
    "Accuracy": [
        accuracy_lr_tuned,
        accuracy_svm_tuned,
        accuracy_mlp
    ]
})

results


,Model,Accuracy
0,Logistic Regression (Tuned),0.964912
1,SVM (Tuned),0.973684
2,Neural Network,0.964912
